In [ ]:
import io
import os
import shutil
import zipfile

# Catalog is passed as a job parameter (default -> ${var.catalog} per target); the default here
# is only a convenience for interactive runs.
dbutils.widgets.text("catalog", "transport_vic_dev")
CATALOG = dbutils.widgets.get("catalog")

# The download task derives export_date from the feed's Last-Modified header and publishes it here.
# debugValue only applies to interactive runs; in a job the upstream task always sets it.
export_date = dbutils.jobs.taskValues.get(
    taskKey="weekly_schedule_download",
    key="export_date",
    debugValue="20260710",
)

# Raw zips and extracted .txt files live in separate volumes, so a reader over gtfs_data never
# has to filter out the archive it came from.
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.`01_bronze`.gtfs_data")

source_base = f"/Volumes/{CATALOG}/01_bronze/gtfs_raw/export_date={export_date}"
destination_base = f"/Volumes/{CATALOG}/01_bronze/gtfs_data/export_date={export_date}"
src_zip = f"{source_base}/gtfs.zip"

# zipfile seeks to read the central directory, and the nested google_transit.zip members need
# random access too. Stage on local ephemeral disk rather than doing that over the volume FUSE mount.
local_zip = f"/tmp/gtfs_{export_date}.zip"
local_out = f"/tmp/gtfs_extract_{export_date}"
shutil.rmtree(local_out, ignore_errors=True)
shutil.copyfile(src_zip, local_zip)

# gtfs.zip holds one numbered folder per PTV mode (1, 2, ... 11), each containing a
# google_transit.zip of the actual .txt feed files. Extract the inner zips, keeping the
# folder names so the mode stays recoverable downstream.
extracted = {}
with zipfile.ZipFile(local_zip) as outer:
    inner_names = [n for n in outer.namelist() if n.endswith("google_transit.zip")]
    if not inner_names:
        raise RuntimeError(f"No google_transit.zip members found in {src_zip}")

    for name in inner_names:
        # "gtfs-schedule/1/google_transit.zip" -> "1"
        folder = os.path.basename(os.path.dirname(name))
        dest_dir = os.path.join(local_out, folder)
        os.makedirs(dest_dir, exist_ok=True)

        # Inner zip must be fully materialised: the outer handle is a forward-only stream.
        with outer.open(name) as f:
            inner_bytes = io.BytesIO(f.read())

        with zipfile.ZipFile(inner_bytes) as inner:
            # Not every mode ships every file (levels.txt / pathways.txt are often absent),
            # so take whatever .txt members are actually present.
            members = [m for m in inner.namelist() if m.endswith(".txt")]
            for m in members:
                with inner.open(m) as s, open(os.path.join(dest_dir, os.path.basename(m)), "wb") as d:
                    shutil.copyfileobj(s, d)

        extracted[folder] = sorted(os.path.basename(m) for m in members)

# Copy the tree up to the volume, mirroring the numbered folder structure.
for folder, files in sorted(extracted.items(), key=lambda kv: int(kv[0])):
    vol_dir = f"{destination_base}/{folder}"
    os.makedirs(vol_dir, exist_ok=True)
    for fn in files:
        shutil.copyfile(os.path.join(local_out, folder, fn), f"{vol_dir}/{fn}")
    print(f"{folder}: {len(files)} files -> {vol_dir}")

shutil.rmtree(local_out, ignore_errors=True)
os.remove(local_zip)

dbutils.jobs.taskValues.set("export_date", export_date)
dbutils.jobs.taskValues.set("mode_folders", sorted(extracted, key=int))
